In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [ ]:
# ================================================================
# Imports
# ================================================================
import numpy as np
from tqdm import tqdm
from scipy.sparse import coo_matrix, diags, bmat, csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial.distance import pdist, squareform
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr

# ================================================================
# Load data & remove isolated
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

# ================================================================
# Keep two largest components
# ================================================================
DIST_TH = 0.22

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords[use_idx]
y = y[use_idx]

S, TT = y.shape
period = 52

print("Using S =", S)

# ================================================================
# Global time trend
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Common MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin
prior_prec = 1.0 / 25.0

# ================================================================
# Functionless block builder
# ================================================================
def run_block(event_name, loc_mask, kappa_builder, save_name):

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    row_idx  = pairs[:, 0]
    time_idx = pairs[:, 1] - 1
    N = len(row_idx)

    next_y = y[pairs[:, 0], pairs[:, 1]]
    kappa  = kappa_builder(next_y)

    t_raw   = time_idx + 1
    t_trend = t_trend_full[time_idx]

    covariates = np.column_stack([
        np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend
    ])

    K = covariates.shape[1]
    theta_dim = K * S

    print(f"{event_name} N =", N)
    print("theta_dim =", theta_dim)

    # ------------------------------------------------------------
    # Design matrix
    # ------------------------------------------------------------
    rows, cols, vals = [], [], []

    for i in range(N):
        s = row_idx[i]
        for k in range(K):
            rows.append(i)
            cols.append(s + k*S)
            vals.append(covariates[i, k])

    X = coo_matrix((vals, (rows, cols)),
                   shape=(N, theta_dim)).tocsr()

    # ------------------------------------------------------------
    # Prior precision
    # ------------------------------------------------------------
    blocks = [
        [prior_prec * diags(np.ones(S)) if i == j else None
         for j in range(K)]
        for i in range(K)
    ]

    curr_prec = bmat(blocks, format="csr")

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------
    all_theta = np.zeros((theta_dim, tot_save))
    curr_theta = np.zeros(theta_dim)
    save_idx = 0

    # ------------------------------------------------------------
    # MCMC
    # ------------------------------------------------------------
    for it in tqdm(range(total_iters), desc=f"MCMC {event_name}"):

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi, size=N)

        XtOmega = X.T.multiply(omega)
        post_prec = (XtOmega @ X + curr_prec).tocsc()

        factor = cholesky(post_prec, mode="simplicial")

        rhs = X.T @ kappa
        mu = factor.solve_A(rhs)

        z = np.random.randn(theta_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_theta = mu + z

        if it >= burn and (it - burn) % thin == 0:
            all_theta[:, save_idx] = curr_theta
            save_idx += 1
            if save_idx == tot_save:
                break

    np.savez_compressed(save_name, all_theta=all_theta)
    print(f"Saved {save_name}")


# ================================================================
# Run p01
# ================================================================
run_block(
    event_name="p01",
    loc_mask=(y[:, :-1] == 0),
    kappa_builder=lambda ny: ny - 0.5,
    save_name="ind01_twoComp_sparse.npz"
)

# ================================================================
# Run p10
# ================================================================
run_block(
    event_name="p10",
    loc_mask=(y[:, :-1] == 1),
    kappa_builder=lambda ny: (1 - ny) - 0.5,
    save_name="ind10_twoComp_sparse.npz"
)

print("Finished both p01 and p10")

Using S = 1557
p01 N = 2716873
theta_dim = 6228


MCMC p01: 100%|█████████▉| 5995/6000 [2:06:59<00:06,  1.27s/it]  


Saved ind01_twoComp_sparse.npz
p10 N = 1491698
theta_dim = 6228


MCMC p10: 100%|█████████▉| 5995/6000 [1:09:52<00:03,  1.43it/s]


Saved ind10_twoComp_sparse.npz
Finished both p01 and p10


In [1]:
# ================================================================
# IID posterior mean log-likelihood (sample-level)
# From scratch: load y -> keep 2 largest components
# -> build p01/p10 -> compute E_post[loglik]
# ================================================================

import numpy as np
import geopandas as gpd
import pyreadr
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path

# ------------------------------------------------
# Config
# ------------------------------------------------
DIST_TH = 0.22
period = 52

BASE_DIR = Path(r"D:\77\Research\temp\snow")

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ------------------------------------------------
# 1️⃣ Load data
# ------------------------------------------------
snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all = snow.iloc[:, 2:].to_numpy()

# ------------------------------------------------
# 2️⃣ Build adjacency & keep 2 largest components
# ------------------------------------------------
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:,0], coords_all[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

y = y_all[use_idx]
S, TT = y.shape

print("Using S =", S, "TT =", TT)

# ------------------------------------------------
# 3️⃣ Global trend
# ------------------------------------------------
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# 🔥 p01
# ================================================================
loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
pairs[:,1] += 1

row_idx  = pairs[:,0]
time_idx = pairs[:,1] - 1
N01 = len(row_idx)

next_y = y[pairs[:,0], pairs[:,1]]
y_vec01 = next_y.astype(float)

t_raw = time_idx + 1
t_trend = t_scaled[time_idx]

cov01 = np.column_stack([
    np.ones(N01),
    np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    t_trend
])

K = cov01.shape[1]
theta_dim = K * S

rows, cols, vals = [], [], []
for i in range(N01):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(cov01[i, k])

X01 = coo_matrix((vals,(rows,cols)), shape=(N01,theta_dim)).tocsr()

theta01 = np.load(BASE_DIR/"ind01_twoComp_sparse.npz")["all_theta"]
M = theta01.shape[1]

loglik_vals01 = np.zeros(M)

for m in tqdm(range(M), desc="IID p01 LLH"):
    theta_m = theta01[:, m]
    phi = X01 @ theta_m
    softplus = np.log1p(np.exp(-np.abs(phi))) + np.maximum(phi,0)
    loglik_vals01[m] = np.sum(y_vec01 * phi - softplus)

llh01 = loglik_vals01.mean()

# ================================================================
# 🔥 p10
# ================================================================
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
pairs[:,1] += 1

row_idx  = pairs[:,0]
time_idx = pairs[:,1] - 1
N10 = len(row_idx)

next_y = y[pairs[:,0], pairs[:,1]]
y_vec10 = (1 - next_y).astype(float)

t_raw = time_idx + 1
t_trend = t_scaled[time_idx]

cov10 = np.column_stack([
    np.ones(N10),
    np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    t_trend
])

rows, cols, vals = [], [], []
for i in range(N10):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(cov10[i, k])

X10 = coo_matrix((vals,(rows,cols)), shape=(N10,theta_dim)).tocsr()

theta10 = np.load(BASE_DIR/"ind10_twoComp_sparse.npz")["all_theta"]
M = theta10.shape[1]

loglik_vals10 = np.zeros(M)

for m in tqdm(range(M), desc="IID p10 LLH"):
    theta_m = theta10[:, m]
    phi = X10 @ theta_m
    softplus = np.log1p(np.exp(-np.abs(phi))) + np.maximum(phi,0)
    loglik_vals10[m] = np.sum(y_vec10 * phi - softplus)

llh10 = loglik_vals10.mean()

# ================================================================
# RESULT
# ================================================================
print("\nIID posterior mean LLH p01:", llh01)
print("IID posterior mean LLH p10:", llh10)
print("TOTAL IID posterior mean LLH:", llh01 + llh10)

Using S = 1557 TT = 2704


IID p10 LLH: 100%|██████████| 1000/1000 [00:42<00:00, 23.40it/s]


IID posterior mean LLH p01: -341292.4556794345
IID posterior mean LLH p10: -269962.526569958
TOTAL IID posterior mean LLH: -611254.9822493925
